# InferScale-X · 01 Core platform on one A100
Runtime → **A100 GPU**. Every step runs as a subprocess (the kernel never imports torch), writes raw results to `benchmarks/results/`, and is safe to re-run. Approximate times are for an A100 40GB.

**Labels you will see in reports:** measured · emulated (single-GPU stand-in) · simulated · extrapolated.

## 0 · Get the code
Upload `InferScale-X.zip` to your Drive root (or to this session), then run:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/InferScale-X'):
    !cp /content/drive/MyDrive/InferScale-X.zip /content/ 2>/dev/null || echo 'upload InferScale-X.zip to /content'
    !cd /content && unzip -q -o InferScale-X.zip
%cd /content/InferScale-X
!nvidia-smi

## 1 · Install (fresh runtime, ~6 min) — writes `requirements/lock-colab.txt`

In [ ]:
!bash scripts/colab_setup.sh core

## 2 · CPU test suite → `tests/results/junit-unit.xml` (~2 min)

In [ ]:
!python -m pytest tests/unit tests/integration/test_gateway_mock.py tests/security tests/failure_injection/test_failure_mock.py --junitxml tests/results/junit-unit.xml

## 3 · Pillar 1 baseline: vLLM direct vs through the gateway (~15 min)

In [ ]:
!python -m benchmarks.runners.exp_baseline --repeats 3

## 4 · Live correctness tests against real vLLM → `junit-live.xml` (~5 min)

In [ ]:
!python scripts/serverctl.py vllm --name vllm-a --port 8001 --model Qwen/Qwen2.5-1.5B-Instruct --gpu-mem 0.85
!python scripts/serverctl.py gateway --backends vllm-a=http://127.0.0.1:8001
!INFERSCALE_LIVE_URL=http://127.0.0.1:9000 python -m pytest tests/integration/test_live_endpoint.py --junitxml tests/results/junit-live.xml
!python scripts/serverctl.py stopall

In [ ]:
!bash scripts/save_results.sh /content/drive/MyDrive/InferScale-X-results

## 5 · Pillar 2 routing: round-robin vs least-outstanding vs KV-aware prefix affinity (~40 min)
Two vLLM replicas share the A100 (0.42 of memory each). Policy deltas are valid; absolute scale is not 2 GPUs.

In [ ]:
!python -m benchmarks.runners.exp_routing --repeats 3

## 6 · Reliability: kill a replica under load, restart it; tenant flood (~10 min)

In [ ]:
!python -m tests.failure_injection.exp_failure --kind vllm

In [ ]:
!bash scripts/save_results.sh /content/drive/MyDrive/InferScale-X-results

## 7 · Pillar 4 KV cache: reuse, HBM pressure, cross-tenant isolation (~45 min)

In [ ]:
!python disaggregation/kv-cache/exp_prefix_cache.py --repeats 3

## 8 · Why disaggregate: prefill→decode interference vs chunk budget (~25 min)

In [ ]:
!python disaggregation/prefill/exp_interference.py --repeats 3

## 9 · KV movement: bandwidth, real KV caches, cost model, emulated disaggregation, NIXL attempt (~10 min)
Optional first: `!pip install nixl` (the script records whether NIXL works here).

In [ ]:
import glob, os
interf = sorted(glob.glob('benchmarks/results/interference/*/report.json'))
arg = f'--interference-report {interf[-1]}' if interf else ''
!python disaggregation/nixl-benchmarks/kv_transfer_bench.py --nixl {arg}

In [ ]:
import glob
kvt = sorted(glob.glob('benchmarks/results/kv_transfer/*/report.json'))
arg = f'--transfer-report {kvt[-1]}' if kvt else ''
!python disaggregation/kv-cache/kv_sim.py {arg} --out benchmarks/results/kv_sim_results.json

## 9b · REAL multi-tier KV: HBM (capped below the working set) vs + CPU DRAM tier vs + localhost remote tier (~40 min)
Tries vLLM native offloading, then OffloadingConnector, then LMCache; records which worked. LMCache is installed with --no-deps so it cannot replace the torch/vLLM build the other sections use; if its compiled extension doesn't match, that variant fails and is recorded.

In [ ]:
!pip install -q --no-deps lmcache || echo 'lmcache install failed - native vLLM offloading will still be tried'
!python disaggregation/kv-cache/exp_kv_offload.py --repeats 3 --remote

## 10 · EXPERIMENTAL: real vLLM prefill/decode split via KV connector on one GPU (~10 min)
Failure is a valid, recorded result.

In [ ]:
!python disaggregation/decode/vllm_disagg_attempt.py

In [ ]:
!bash scripts/save_results.sh /content/drive/MyDrive/InferScale-X-results

## 11 · Pillar 3: quantization BF16 vs AWQ-INT4 vs GPTQ-INT8 vs FP8 weight-only (~40 min)

In [ ]:
!python optimization/quantization/exp_quant.py --repeats 3

## 12 · Custom Triton kernel: fused residual-add + RMSNorm (~3 min)

In [ ]:
!python optimization/triton-kernels/fused_add_rmsnorm.py --profile
!which nsys && nsys profile -o benchmarks/results/triton_kernel/nsys_fused --force-overwrite true python optimization/triton-kernels/fused_add_rmsnorm.py --iters 20 || echo 'nsys not available on this runtime - torch.profiler trace saved instead'

## 12b · Custom Triton FlashAttention-style forward vs PyTorch SDPA (~5 min)

In [ ]:
!python optimization/triton-kernels/flash_attention.py

## 13 · Speculative decoding: HF lossless check + vLLM draft/n-gram at concurrency 1 and 16 (~40 min)

In [ ]:
!python optimization/speculative-decoding/exp_specdec.py --n 30 --vllm --repeats 2

In [ ]:
!bash scripts/save_results.sh /content/drive/MyDrive/InferScale-X-results

## 14 · Pillar 5 agents: Qwen2.5-7B through the gateway; agent vs RAG; security suites (~20 min)

In [ ]:
!python scripts/serverctl.py vllm --name vllm-a --port 8001 --model Qwen/Qwen2.5-7B-Instruct --gpu-mem 0.9 --max-len 8192
!python scripts/serverctl.py gateway --backends vllm-a=http://127.0.0.1:8001
!python agents/eval/run_eval.py --url http://127.0.0.1:9000 --model Qwen/Qwen2.5-7B-Instruct
!python scripts/serverctl.py stopall

## 14b · Retrieval quality: BM25 vs dense vs hybrid (recall@k, MRR, nDCG) (~3 min)

In [ ]:
!pip install -q sentence-transformers
!python agents/eval/retrieval_eval.py

## 15 · vLLM entry for the engine comparison (SGLang / TRT-LLM run in notebook 03) (~20 min)

In [ ]:
!python inference/exp_engines.py --engine vllm --repeats 3

## 16 · Optional: live monitoring (Prometheus + GPU exporter) while a load test runs

In [ ]:
!mkdir -p .bin && cd .bin && [ -x prometheus ] || (wget -q https://github.com/prometheus/prometheus/releases/download/v2.53.2/prometheus-2.53.2.linux-amd64.tar.gz && tar xzf prometheus-2.53.2.linux-amd64.tar.gz && cp prometheus-2.53.2.linux-amd64/prometheus .)
!nohup python infrastructure/monitoring/gpu_exporter.py --port 9401 > logs/gpu_exporter.log 2>&1 &
!nohup .bin/prometheus --config.file=infrastructure/monitoring/prometheus.yml --storage.tsdb.path=/tmp/prom > logs/prometheus.log 2>&1 &
!python scripts/serverctl.py vllm --name vllm-a --port 8001 --model Qwen/Qwen2.5-1.5B-Instruct
!python scripts/serverctl.py gateway --backends vllm-a=http://127.0.0.1:8001
!python -m benchmarks.runners.loadgen --url http://127.0.0.1:9000 --api-key sk-dev-acme --workload benchmarks/workloads/multitenant.yaml --tenant-keys acme=sk-dev-acme,globex=sk-dev-globex,freetier=sk-dev-free --out benchmarks/results/multitenant --repeats 1
!curl -s 'http://127.0.0.1:9090/api/v1/query?query=histogram_quantile(0.95,sum(rate(inferscale_ttft_seconds_bucket[5m]))by(le))' | head -c 600
!python scripts/serverctl.py stopall; pkill -f prometheus; pkill -f gpu_exporter

## 17 · Plots, readiness checklist, JD coverage, claims, results page, capacity model, save

In [ ]:
!python benchmarks/analysis/make_plots.py
!python scripts/evidence_report.py
!python scripts/publish_results.py
!cat docs/CLAIMS.md
!head -30 docs/jd-coverage.md

In [ ]:
import glob
base = sorted(glob.glob('benchmarks/results/baseline/*/direct/aggregate.json'))
if base:
    !python docs/capacity-planning/capacity_model.py --preset qwen2.5-1.5b --gpu a100-40gb --measured {base[-1]} --rps 50 --in-tokens 1000 --out-tokens 200

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('benchmarks/results/plots/*.png')):
    print(p); display(Image(p))

In [ ]:
!bash scripts/save_results.sh /content/drive/MyDrive/InferScale-X-results

## 18 · Optional: live public demo endpoint (real model on this A100, fresh API key, Cloudflare quick tunnel)
Prints a temporary HTTPS URL + key you can hit from your Mac or share for a demo. Stop it when done.

In [ ]:
!bash scripts/colab_public_endpoint.sh Qwen/Qwen2.5-1.5B-Instruct

In [ ]:
!bash scripts/colab_public_endpoint.sh stop